#Import

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#Create Flag Parameter

In [0]:
# Check whether the Gold dimension table already exists
if spark.catalog.tableExists("store_catalog.gold.dim_crm_cust_info"):
    incremental_flag = "1"
else:
    incremental_flag = "0"

In [0]:
print(incremental_flag)

1


#Create Dimension crm_cust_info

###Fetch Relative Columns

In [0]:
# Load distinct customer records from the Silver layer
df_scr = spark.sql('''
        SELECT DISTINCT * FROM parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_cust_info` 
    ''')


In [0]:
df_scr.limit(10).display()

customer_id,customer_key,first_name,last_name,marital_status,gender,create_date,record_priority
11065,AW00011065,Jessica,Henderson,Married,Female,2025-10-07,1
11082,AW00011082,Angela,Butler,Single,Female,2025-10-07,1
11414,AW00011414,Ian,Richardson,Single,n/a,2025-10-08,1
12211,AW00012211,Victor,Romero,Married,n/a,2025-10-08,1
12350,AW00012350,Logan,Hernandez,Married,n/a,2025-10-08,1
12365,AW00012365,Alex,Evans,Married,n/a,2025-10-08,1
12386,AW00012386,Katherine,Perry,Single,n/a,2025-10-08,1
12528,AW00012528,Devin,Wilson,Single,n/a,2025-10-08,1
12688,AW00012688,Stacy,Navarro,Single,n/a,2025-10-12,1
13011,AW00013011,Anne,Ortega,Single,n/a,2025-10-12,1


###dim_crm_cust_info - Initial and Incremental

In [0]:
# Read existing Gold records or create an empty DataFrame for the initial load
if spark.catalog.tableExists("store_catalog.gold.dim_crm_cust_info") == True:
    df_sink = spark.sql('''
            SELECT
                crm_cust_info_surrogate_key,
                customer_id,
                customer_key,
                first_name,
                last_name,
                marital_status,
                gender,
                create_date
            FROM
                store_catalog.gold.dim_crm_cust_info
        ''')

else:
    df_sink = spark.sql('''
            SELECT
                1 as crm_cust_info_surrogate_key,
                customer_id,
                customer_key,
                first_name,
                last_name,
                marital_status,
                gender,
                create_date
            FROM
                parquet.`abfss://silver@storenapatdatalake.dfs.core.windows.net/crm_cust_info`
            WHERE
                1 = 0 
        ''')


###Filtering new records and old records

In [0]:
# Match Silver records with existing Gold records by customer ID
df_filter = df_scr.join(df_sink,df_scr["customer_id"] == df_sink["customer_id"], "left")\
        .select(df_scr["customer_id"],df_scr["customer_key"],df_scr["first_name"],
            df_scr["last_name"],df_scr["marital_status"],df_scr["gender"],
            df_scr["create_date"],df_sink["crm_cust_info_surrogate_key"]
    )

In [0]:
df_filter.groupBy("customer_id") \
    .count() \
    .filter(col("count") > 1) \
    .limit(10)\
    .show(truncate=False)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



**df_filter_old**

In [0]:
# Separate existing customer records
df_filter_old = df_filter.filter(col("crm_cust_info_surrogate_key").isNotNull())

**df_filter_new**

In [0]:
# Select new customers that require surrogate keys
df_filter_new = df_filter.filter(col("crm_cust_info_surrogate_key").isNull())\
    .select(df_scr["customer_id"],df_scr["customer_key"],df_scr["first_name"],
            df_scr["last_name"],df_scr["marital_status"],df_scr["gender"],
            df_scr["create_date"]
    )

In [0]:
df_filter_new.limit(10).display()

customer_id,customer_key,first_name,last_name,marital_status,gender,create_date


###Create Surrogate Key

**Fetch the max Surrogate Key from existing table**

In [0]:
# Retrieve the next available surrogate key
if (incremental_flag == "0"):
    max_value = 1
else:
    max_value_df = spark.sql('''SELECT MAX(crm_cust_info_surrogate_key) FROM store_catalog.gold.dim_crm_cust_info''')
    max_value = max_value_df.collect()[0][0]+1

In [0]:
print("incremental_flag:", incremental_flag)
print("max_value:", max_value)

incremental_flag: 1
max_value: 18485


**Create Surrogate Key column and Add the max surrogate key**

In [0]:
# Assign unique surrogate keys to new customer records
df_filter_new = df_filter_new.withColumn("crm_cust_info_surrogate_key",max_value + monotonically_increasing_id())
df_filter_new.limit(10).display()

customer_id,customer_key,first_name,last_name,marital_status,gender,create_date,crm_cust_info_surrogate_key


###Create Final DF - df_filter_old + df_filter_new

In [0]:
# Combine new customers with existing customer records
df_final = df_filter_new.union(df_filter_old)

In [0]:
df_final.limit(10).display()

customer_id,customer_key,first_name,last_name,marital_status,gender,create_date,crm_cust_info_surrogate_key
11065,AW00011065,Jessica,Henderson,Married,Female,2025-10-07,1
11082,AW00011082,Angela,Butler,Single,Female,2025-10-07,2
11414,AW00011414,Ian,Richardson,Single,n/a,2025-10-08,3
12211,AW00012211,Victor,Romero,Married,n/a,2025-10-08,4
12350,AW00012350,Logan,Hernandez,Married,n/a,2025-10-08,5
12365,AW00012365,Alex,Evans,Married,n/a,2025-10-08,6
12386,AW00012386,Katherine,Perry,Single,n/a,2025-10-08,7
12528,AW00012528,Devin,Wilson,Single,n/a,2025-10-08,8
12688,AW00012688,Stacy,Navarro,Single,n/a,2025-10-12,9
13011,AW00013011,Anne,Ortega,Single,n/a,2025-10-12,10


# SCD TYPE - 1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:

if spark.catalog.tableExists("store_catalog.gold.dim_crm_cust_info"): 
    # Incremental Run
    # Merge customer records into the Gold dimension table
    delta_tbl = DeltaTable.forPath(spark,"abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_crm_cust_info")

    delta_tbl.alias("trg").merge(df_final.alias("src"),"trg.crm_cust_info_surrogate_key = src.crm_cust_info_surrogate_key")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()
else: 
    # Initial Run
    # Create the Gold dimension table during the initial load
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@storenapatdatalake.dfs.core.windows.net/dim_crm_cust_info")\
        .saveAsTable("store_catalog.gold.dim_crm_cust_info")


In [0]:
%sql
SELECT * FROM store_catalog.gold.dim_crm_cust_info ORDER BY crm_cust_info_surrogate_key DESC LIMIT 10

customer_id,customer_key,first_name,last_name,marital_status,gender,create_date,crm_cust_info_surrogate_key
29467,AW00029467,Monica,Mehta,Married,Female,2026-01-25,18484
29458,AW00029458,Danny,Moreno,Single,Male,2026-01-25,18483
29343,AW00029343,Glenn,Wu,Married,Male,2026-01-25,18482
29305,AW00029305,Beth,Alvarez,Single,Female,2026-01-25,18481
29252,AW00029252,Alex,Parker,Married,Male,2026-01-25,18480
29190,AW00029190,Kaitlin,Suri,Single,Female,2026-01-25,18479
28759,AW00028759,Melissa,Rivera,Single,Female,2026-01-25,18478
28666,AW00028666,Kelvin,Zhu,Married,Male,2026-01-25,18477
28432,AW00028432,Bruce,Carlson,Single,n/a,2026-01-25,18476
28165,AW00028165,Alexa,Rivera,Married,Female,2026-01-25,18475


In [0]:
%sql
SELECT COUNT(*) FROM store_catalog.gold.dim_crm_cust_info

count(1)
18484
